In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np


In [2]:
df = pd.read_csv('/content/insurance.csv')
df.sample(5)

,age,sex,bmi,children,smoker,region,expenses
777,45,male,39.8,0,no,northeast,7448.40
1191,41,female,21.8,1,no,northeast,13725.47
900,49,male,22.5,0,no,northeast,8688.86
855,20,female,29.6,0,no,southwest,1875.34
1166,57,male,40.4,0,no,southeast,10982.50


In [3]:
df_feat =  df.copy()
#since we will apply feature engg so we created a copy of this file

In [4]:
df.shape

(1338, 7)

In [5]:
list(df.columns)

['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'expenses']

In [6]:
#Feat 2: Age
def age_group(age):
  if age < 25:
    return "young"
  elif age < 35:
    return "adult"
  elif age < 55:
    return "middle_aged"
  return "senior"

In [7]:
df_feat["age_group"] = df_feat['age'].apply(age_group)

In [8]:
#feat 3: lifestyle risk through smoking
def lifestyle_risk(row):
  if row["smoker"] and row["bmi"] > 30:
    return "High"
  elif row["smoker"] or row["bmi"] > 27:
    return "medium"
  else:
    return "low"

In [9]:
df_feat["lifestyle_risk"]=df_feat.apply(lifestyle_risk, axis=1)


In [10]:
tier_1_cities = ['southwest','southeast']
tier_2_cities = ['northwest','northeast']

In [11]:
# feat 4: city tier
def city_tier(region):
    if region in tier_1_cities:
        return 1
    elif region in tier_2_cities:
        return 2
    else:
        return 3

In [12]:
df_feat["city_tier"] = df_feat["region"].apply(city_tier)

In [13]:
df_feat.drop(columns=['age','smoker','region'])[['expenses','bmi','age_group','children','sex','lifestyle_risk','city_tier']]

,expenses,bmi,age_group,children,sex,lifestyle_risk,city_tier
0,16884.92,27.9,young,0,female,medium,1
1,1725.55,33.8,young,1,male,High,1
2,4449.46,33.0,adult,3,male,High,1
3,21984.47,22.7,adult,0,male,medium,2
4,3866.86,28.9,adult,0,male,medium,2
...,...,...,...,...,...,...,...
1333,10600.55,31.0,middle_aged,3,male,High,2
1334,2205.98,31.9,young,0,female,High,2
1335,1629.83,36.9,young,0,female,High,1
1336,2007.95,25.8,young,0,female,medium,1


In [14]:
#select features and target
X = df_feat[['bmi','age_group','children','sex','lifestyle_risk','city_tier']]
y=df_feat['expenses']

In [15]:
X

,bmi,age_group,children,sex,lifestyle_risk,city_tier
0,27.9,young,0,female,medium,1
1,33.8,young,1,male,High,1
2,33.0,adult,3,male,High,1
3,22.7,adult,0,male,medium,2
4,28.9,adult,0,male,medium,2
...,...,...,...,...,...,...
1333,31.0,middle_aged,3,male,High,2
1334,31.9,young,0,female,High,2
1335,36.9,young,0,female,High,1
1336,25.8,young,0,female,medium,1


In [16]:
y

,expenses
0,16884.92
1,1725.55
2,4449.46
3,21984.47
4,3866.86
...,...
1333,10600.55
1334,2205.98
1335,1629.83
1336,2007.95


In [17]:
#define categorical and numeric features
categorical_features = ["age_group","sex","lifestyle_risk","city_tier"]
numerical_features = ["bmi","children"]

In [18]:
print(df_feat.columns.tolist())

['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'expenses', 'age_group', 'lifestyle_risk', 'city_tier']


In [19]:
#create column transformer for one hot encoding
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(),categorical_features),
        ("num","passthrough",numerical_features)
    ]
)

In [20]:
#create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("classifier",RandomForestClassifier(random_state=42))
])

In [22]:
print(pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group', 'sex',
                                                   'lifestyle_risk',
                                                   'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'children'])])),
                ('classifier', RandomForestClassifier(random_state=42))])


Tutorial show classifiaction but since we're using diffrent dataset so i had to use regression


In [23]:
from sklearn.ensemble import RandomForestRegressor

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

In [24]:
# Split data and train model
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group', 'sex',
                                                   'lifestyle_risk',
                                                   'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'children'])])),
                ('regressor', RandomForestRegressor(random_state=42))])

In [26]:
#pred and evaluate
#y_pred = pipeline.predict(X_test)
#accuracy_score(y_test, y_pred)
from sklearn.metrics import r2_score

y_pred = pipeline.predict(X_test)
print(r2_score(y_test, y_pred))

-0.10330270935775432


In [27]:
X_test.sample(5)

,bmi,age_group,children,sex,lifestyle_risk,city_tier
1040,28.0,middle_aged,0,female,medium,2
953,30.2,middle_aged,2,male,High,1
148,37.4,middle_aged,1,female,High,2
616,28.6,senior,0,female,medium,2
73,32.0,senior,1,male,High,1


In [28]:
import pickle
#save the trained pipeline using pickle
pickle_model_path = "model.pkl"
with open(pickle_model_path, "wb") as f:
  pickle.dump(pipeline, f)